# 🚀 Setup Workshop — Data Engineering & Data Analytics

**Este notebook configura todo o ambiente necessário para o workshop.**

Execute todas as células em ordem. O script irá:
1. Carregar a configuração de catalog/schema (via `00_Config`)
2. Criar o catalog e schema (se necessário)
3. Importar os dados reais de tickets de TI a partir dos CSVs na pasta `data/`
4. Criar as tabelas Bronze (tb_tickets, tb_agents, tb_sla, tb_tickets_comments)
5. Criar o volume `documentos_normas` e carregar os PDFs normativos
6. Criar a Skill de padronização para o Genie Code

**Pré-requisitos:**
- Permissão de CREATE CATALOG ou acesso a um catalog existente
- Compute Serverless ou cluster com acesso ao Unity Catalog

**Configuração:** Edite o notebook `00_Config` para definir catalog e schema do seu ambiente.

In [0]:
%run ./00_Config

In [0]:
# ==========================================================
# Utilização do Catalog e Schema
# Se já existirem, não faz nada (IF NOT EXISTS)
# ==========================================================

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print(f"✅ Catalog '{CATALOG}' e schema '{SCHEMA}' prontos")

In [0]:
import pandas as pd
from pyspark.sql.types import *

# ==========================================================
# Importação dos dados reais a partir dos CSVs
# Os CSVs estão na pasta data/ do workshop (versionados no git)
# ==========================================================

# Schemas explícitos para garantir tipos corretos
schemas = {
    "tb_tickets": StructType([
        StructField("ticket_id", LongType()),
        StructField("status", StringType()),
        StructField("priority", StringType()),
        StructField("source", StringType()),
        StructField("topic", StringType()),
        StructField("created_time", TimestampType()),
        StructField("close_time", TimestampType()),
        StructField("product_group", StringType()),
        StructField("support_level", StringType()),
        StructField("country", StringType()),
        StructField("latitude", DoubleType()),
        StructField("longitude", DoubleType())
    ]),
    "tb_agents": StructType([
        StructField("ticket_id", LongType()),
        StructField("agent_group", StringType()),
        StructField("agent_name", StringType()),
        StructField("agent_interactions", DoubleType())
    ]),
    "tb_sla": StructType([
        StructField("ticket_id", LongType()),
        StructField("expected_sla_to_resolve", TimestampType()),
        StructField("expected_sla_to_first_response", TimestampType()),
        StructField("first_response_time", TimestampType()),
        StructField("sla_for_first_response", StringType()),
        StructField("resolution_time", TimestampType()),
        StructField("sla_for_resolution", StringType()),
        StructField("survey_results", DoubleType())
    ]),
    "tb_tickets_comments": StructType([
        StructField("ticket_id", LongType()),
        StructField("comments", StringType())
    ])
}

for table_name, schema in schemas.items():
    csv_path = f"{DATA_DIR}/{table_name}.csv"
    pdf = pd.read_csv(csv_path)
    for field in schema.fields:
        if isinstance(field.dataType, TimestampType) and field.name in pdf.columns:
            pdf[field.name] = pd.to_datetime(pdf[field.name])
    df = spark.createDataFrame(pdf, schema=schema)
    df.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{table_name}")
    print(f"✅ {table_name}: {df.count()} registros importados de {table_name}.csv")

In [0]:
import shutil, os

# ==========================================================
# Criar Volume para documentos normativos (PDFs)
# e fazer upload dos arquivos da pasta data/pdf
# ==========================================================

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.documentos_normas")
spark.sql(f"""
  COMMENT ON VOLUME {CATALOG}.{SCHEMA}.documentos_normas IS
  'Documentos normativos de TI: SLA e Níveis de Serviço, Processos de Escalação, Segurança e Controle de Acesso, Qualidade e Padrões de Atendimento, Gestão de Mudanças e CMDB, Continuidade de Serviço e DR.'
""")

pdf_source_dir = f"{DATA_DIR}/pdf"
volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/documentos_normas"

pdf_files = [f for f in os.listdir(pdf_source_dir) if f.lower().endswith('.pdf')]

for pdf_file in sorted(pdf_files):
    src = os.path.join(pdf_source_dir, pdf_file)
    dst = os.path.join(volume_path, pdf_file)
    shutil.copy2(src, dst)
    print(f"✅ {pdf_file} → {volume_path}/")

print(f"\n📁 {len(pdf_files)} PDFs carregados no volume {CATALOG}.{SCHEMA}.documentos_normas")

In [0]:
%sql
-- ==========================================================
-- Validação: Contagem das tabelas Bronze criadas
-- ==========================================================
SELECT 'tb_tickets' AS tabela, COUNT(*) AS registros FROM tb_tickets
UNION ALL
SELECT 'tb_agents', COUNT(*) FROM tb_agents
UNION ALL
SELECT 'tb_sla', COUNT(*) FROM tb_sla
UNION ALL
SELECT 'tb_tickets_comments', COUNT(*) FROM tb_tickets_comments;